# 1.4 — Reportes Automaticos

## Unidad 1: Introduccion a Desarrollo de Productos de Datos

Un reporte que se genera manualmente cada semana no es un producto de datos — es una tarea repetitiva. En este notebook aprenderemos a construir reportes que se generan solos: desde parametrizar notebooks con Papermill, hasta crear documentos profesionales con Jinja2 y exportarlos a PDF.

### Contenido:
1. Notebooks parametrizados con Papermill
2. Plantillas HTML con Jinja2
3. Exportacion a PDF con weasyprint
4. Automatizacion del flujo
5. Distribucion por correo

In [ ]:
# ============================================================
# INSTALACION DE DEPENDENCIAS
# ============================================================

# Ejecutar solo la primera vez
# papermill: ejecuta notebooks con parametros distintos
# jinja2: motor de plantillas para generar HTML dinamico
# weasyprint: convierte HTML a PDF

# !pip install papermill jinja2 weasyprint openpyxl matplotlib

In [ ]:
# ============================================================
# IMPORTACIONES Y DATOS DE EJEMPLO
# ============================================================

import pandas as pd
import numpy as np
import os
from datetime import datetime

# Crear un dataset de ventas para usar en todo el notebook
np.random.seed(42)
n = 500

productos = ["Dashboard", "Reporte", "API", "App Web"]
regiones = ["Eje Cafetero", "Bogota", "Medellin", "Cali", "Barranquilla"]
vendedores = ["Ana Garcia", "Carlos Lopez", "Maria Ruiz", "Pedro Martin"]

df = pd.DataFrame({
    "fecha": pd.date_range("2024-01-01", periods=n, freq="D"),
    "producto": np.random.choice(productos, n),
    "region": np.random.choice(regiones, n),
    "vendedor": np.random.choice(vendedores, n),
    "unidades": np.random.randint(1, 50, n),
    "precio_unitario": np.round(np.random.uniform(100, 800, n), 2),
})

df["ingreso"] = df["unidades"] * df["precio_unitario"]
df["mes"] = df["fecha"].dt.to_period("M")

# Guardar para que Papermill lo lea despues
df.to_csv("ventas_completas.csv", index=False)
print(f"Dataset: {df.shape[0]} filas, {df.shape[1]} columnas")
df.head()

---
## 1. Notebooks parametrizados con Papermill

Papermill resuelve un problema concreto: tienes un notebook de analisis que funciona bien, pero necesitas ejecutarlo con datos distintos — otro mes, otra region, otro producto. En lugar de abrir el notebook y cambiar variables a mano, Papermill lo ejecuta desde la terminal inyectando los parametros que necesites.

### 1.1 Crear el notebook plantilla

El primer paso es crear un notebook que funcione como plantilla. La unica diferencia con un notebook normal es que tiene una celda marcada con el tag `parameters` — ahi van las variables que Papermill podra cambiar.

In [ ]:
# ============================================================
# CREAR EL NOTEBOOK PLANTILLA PROGRAMATICAMENTE
# ============================================================

# Vamos a generar el notebook plantilla desde codigo
# para que quede completo y funcional

import json

# Estructura de un archivo .ipynb: es un JSON con celdas
notebook_plantilla = {
    "cells": [
        {
            # Celda markdown: titulo
            "cell_type": "markdown",
            "metadata": {},
            "source": ["# Reporte de Ventas\n", "\n", "Generado automaticamente."]
        },
        {
            # Celda de parametros: esta es la que Papermill busca
            # El tag "parameters" va en metadata.tags
            "cell_type": "code",
            "metadata": {
                "tags": ["parameters"]  # <-- Esto es lo que Papermill necesita
            },
            "source": [
                "# Parametros por defecto\n",
                "# Papermill reemplaza estos valores al ejecutar\n",
                "region = 'Bogota'\n",
                "mes_inicio = '2024-01'\n",
                "mes_fin = '2024-03'\n",
                "archivo_datos = 'ventas_completas.csv'"
            ],
            "execution_count": None,
            "outputs": []
        },
        {
            # Celda de carga y filtrado
            "cell_type": "code",
            "metadata": {},
            "source": [
                "import pandas as pd\n",
                "import matplotlib.pyplot as plt\n",
                "\n",
                "# Cargar datos\n",
                "df = pd.read_csv(archivo_datos, parse_dates=['fecha'])\n",
                "df['mes'] = df['fecha'].dt.to_period('M').astype(str)\n",
                "\n",
                "# Filtrar por region y rango de meses\n",
                "mask = (\n",
                "    (df['region'] == region) &\n",
                "    (df['mes'] >= mes_inicio) &\n",
                "    (df['mes'] <= mes_fin)\n",
                ")\n",
                "df_filtrado = df[mask].copy()\n",
                "\n",
                "print(f'Region: {region}')\n",
                "print(f'Periodo: {mes_inicio} a {mes_fin}')\n",
                "print(f'Registros: {len(df_filtrado)}')"
            ],
            "execution_count": None,
            "outputs": []
        },
        {
            # Celda de resumen
            "cell_type": "code",
            "metadata": {},
            "source": [
                "# Resumen por producto\n",
                "resumen = df_filtrado.groupby('producto').agg(\n",
                "    ventas=('ingreso', 'count'),\n",
                "    ingreso_total=('ingreso', 'sum'),\n",
                "    ingreso_promedio=('ingreso', 'mean'),\n",
                "    unidades_total=('unidades', 'sum')\n",
                ").round(2).sort_values('ingreso_total', ascending=False)\n",
                "\n",
                "print(f'Ingreso total: ${df_filtrado[\"ingreso\"].sum():,.0f}')\n",
                "print(f'Ventas totales: {len(df_filtrado)}')\n",
                "print()\n",
                "resumen"
            ],
            "execution_count": None,
            "outputs": []
        },
        {
            # Celda de grafico
            "cell_type": "code",
            "metadata": {},
            "source": [
                "# Grafico de ingreso por producto\n",
                "fig, ax = plt.subplots(figsize=(8, 4))\n",
                "resumen['ingreso_total'].plot(kind='barh', ax=ax, color='#2563eb')\n",
                "ax.set_title(f'Ingreso por Producto — {region}')\n",
                "ax.set_xlabel('Ingreso Total ($)')\n",
                "plt.tight_layout()\n",
                "plt.savefig(f'grafico_{region.lower().replace(\" \", \"_\")}.png', dpi=150)\n",
                "plt.show()"
            ],
            "execution_count": None,
            "outputs": []
        }
    ],
    "metadata": {
        "kernelspec": {
            "display_name": "Python 3",
            "language": "python",
            "name": "python3"
        },
        "language_info": {"name": "python", "version": "3.10.0"}
    },
    "nbformat": 4,
    "nbformat_minor": 4
}

# Guardar el notebook plantilla
with open("reporte_plantilla.ipynb", "w") as f:
    json.dump(notebook_plantilla, f, indent=1)

print("Archivo creado: reporte_plantilla.ipynb")
print("Este notebook tiene una celda con tag 'parameters' que Papermill puede inyectar.")

### 1.2 Ejecutar con Papermill

In [ ]:
# ============================================================
# EJECUTAR EL NOTEBOOK CON PARAMETROS DISTINTOS
# ============================================================

import papermill as pm

# pm.execute_notebook() toma:
#   - input_path: notebook plantilla
#   - output_path: donde guardar el resultado ejecutado
#   - parameters: diccionario con los valores a inyectar

# Generar reporte para Bogota, primer trimestre 2024
pm.execute_notebook(
    input_path="reporte_plantilla.ipynb",
    output_path="reporte_bogota_Q1.ipynb",
    parameters={
        "region": "Bogota",
        "mes_inicio": "2024-01",
        "mes_fin": "2024-03",
        "archivo_datos": "ventas_completas.csv"
    }
)

print("Reporte generado: reporte_bogota_Q1.ipynb")

In [ ]:
# ============================================================
# GENERAR REPORTES EN LOTE — uno por region
# ============================================================

# Esto es el poder real de Papermill:
# un loop que genera N reportes con una sola plantilla

regiones = ["Bogota", "Medellin", "Eje Cafetero", "Cali", "Barranquilla"]

for region in regiones:
    # Nombre del archivo de salida limpio
    nombre_archivo = f"reporte_{region.lower().replace(' ', '_')}_Q1.ipynb"
    
    try:
        pm.execute_notebook(
            input_path="reporte_plantilla.ipynb",
            output_path=nombre_archivo,
            parameters={
                "region": region,
                "mes_inicio": "2024-01",
                "mes_fin": "2024-03",
                "archivo_datos": "ventas_completas.csv"
            }
        )
        print(f"  Generado: {nombre_archivo}")
    except Exception as e:
        print(f"  Error en {region}: {e}")

print(f"\nTotal: {len(regiones)} reportes generados")

In [ ]:
# ============================================================
# CONVERTIR NOTEBOOK A HTML (alternativa a PDF)
# ============================================================

# nbconvert transforma un notebook ejecutado a HTML, PDF o slides
# Se puede usar desde la terminal o desde Python

# Desde la terminal seria:
# jupyter nbconvert --to html reporte_bogota_Q1.ipynb

# Desde Python:
import subprocess

resultado = subprocess.run(
    ["jupyter", "nbconvert", "--to", "html", "reporte_bogota_Q1.ipynb"],
    capture_output=True,
    text=True
)

if resultado.returncode == 0:
    print("Convertido: reporte_bogota_Q1.html")
else:
    print(f"Error: {resultado.stderr}")

---
## 2. Plantillas HTML con Jinja2

Papermill es ideal para reportes tecnicos (notebooks ejecutados). Pero cuando el destinatario es un gerente o un cliente, necesitamos un documento con diseno profesional. Jinja2 nos permite crear plantillas HTML donde inyectamos datos, tablas y graficos dinamicamente.

### 2.1 Crear la plantilla HTML

In [ ]:
# ============================================================
# PLANTILLA HTML CON JINJA2
# ============================================================

# Una plantilla Jinja2 es un archivo HTML con marcadores especiales:
#   {{ variable }}      -> inserta el valor de una variable
#   {% for item in lista %} -> bucle
#   {% if condicion %}  -> condicional

plantilla_html = """
<!DOCTYPE html>
<html lang="es">
<head>
    <meta charset="UTF-8">
    <title>{{ titulo }}</title>
    <style>
        /* Estilos del reporte */
        body {
            font-family: 'Segoe UI', Arial, sans-serif;
            margin: 40px;
            color: #1a1a1a;
            font-size: 14px;
            line-height: 1.6;
        }
        
        .header {
            border-bottom: 3px solid #2563eb;
            padding-bottom: 15px;
            margin-bottom: 30px;
        }
        
        .header h1 {
            color: #2563eb;
            margin-bottom: 5px;
        }
        
        .header .subtitulo {
            color: #666;
            font-size: 16px;
        }
        
        .metricas {
            display: flex;
            gap: 20px;
            margin: 25px 0;
        }
        
        .metrica-card {
            flex: 1;
            background: #f0f4ff;
            border-radius: 8px;
            padding: 20px;
            text-align: center;
        }
        
        .metrica-card .valor {
            font-size: 28px;
            font-weight: bold;
            color: #2563eb;
        }
        
        .metrica-card .etiqueta {
            color: #666;
            font-size: 13px;
            margin-top: 5px;
        }
        
        table {
            width: 100%;
            border-collapse: collapse;
            margin: 20px 0;
        }
        
        th {
            background: #2563eb;
            color: white;
            padding: 10px 15px;
            text-align: left;
        }
        
        td {
            padding: 8px 15px;
            border-bottom: 1px solid #e5e7eb;
        }
        
        tr:nth-child(even) {
            background: #f9fafb;
        }
        
        .grafico {
            text-align: center;
            margin: 30px 0;
        }
        
        .grafico img {
            max-width: 100%;
            border-radius: 8px;
            box-shadow: 0 2px 8px rgba(0,0,0,0.1);
        }
        
        .footer {
            margin-top: 40px;
            padding-top: 15px;
            border-top: 1px solid #e5e7eb;
            color: #999;
            font-size: 12px;
        }
    </style>
</head>
<body>

    <!-- Cabecera -->
    <div class="header">
        <h1>{{ titulo }}</h1>
        <div class="subtitulo">{{ subtitulo }}</div>
    </div>

    <!-- Tarjetas de metricas -->
    <div class="metricas">
        {% for metrica in metricas %}
        <div class="metrica-card">
            <div class="valor">{{ metrica.valor }}</div>
            <div class="etiqueta">{{ metrica.etiqueta }}</div>
        </div>
        {% endfor %}
    </div>

    <!-- Tabla de detalle -->
    <h2>Detalle por Producto</h2>
    <table>
        <thead>
            <tr>
                {% for col in columnas_tabla %}
                <th>{{ col }}</th>
                {% endfor %}
            </tr>
        </thead>
        <tbody>
            {% for fila in filas_tabla %}
            <tr>
                {% for valor in fila %}
                <td>{{ valor }}</td>
                {% endfor %}
            </tr>
            {% endfor %}
        </tbody>
    </table>

    <!-- Grafico -->
    {% if ruta_grafico %}
    <div class="grafico">
        <h2>Distribucion de Ingresos</h2>
        <img src="{{ ruta_grafico }}" alt="Grafico de ingresos">
    </div>
    {% endif %}

    <!-- Pie de pagina -->
    <div class="footer">
        Reporte generado automaticamente el {{ fecha_generacion }}
    </div>

</body>
</html>
"""

# Guardar la plantilla en disco
with open("plantilla_reporte.html", "w", encoding="utf-8") as f:
    f.write(plantilla_html)

print("Plantilla creada: plantilla_reporte.html")
print("Variables disponibles: titulo, subtitulo, metricas, columnas_tabla, filas_tabla, ruta_grafico, fecha_generacion")

### 2.2 Renderizar la plantilla con datos

In [ ]:
# ============================================================
# PREPARAR LOS DATOS PARA LA PLANTILLA
# ============================================================

import matplotlib
matplotlib.use("Agg")  # Backend sin GUI, para guardar imagenes sin mostrar ventana
import matplotlib.pyplot as plt

# Parametros del reporte
region_reporte = "Bogota"
periodo = "Enero - Junio 2024"

# Filtrar datos
df_region = df[df["region"] == region_reporte].copy()

# --- Metricas principales ---
ingreso_total = df_region["ingreso"].sum()
n_ventas = len(df_region)
ticket_promedio = df_region["ingreso"].mean()
producto_top = df_region.groupby("producto")["ingreso"].sum().idxmax()

metricas = [
    {"valor": f"${ingreso_total:,.0f}", "etiqueta": "Ingreso Total"},
    {"valor": f"{n_ventas}", "etiqueta": "Total Ventas"},
    {"valor": f"${ticket_promedio:,.0f}", "etiqueta": "Ticket Promedio"},
    {"valor": producto_top, "etiqueta": "Producto Lider"},
]

# --- Tabla resumen ---
resumen = df_region.groupby("producto").agg(
    Ventas=("ingreso", "count"),
    Ingreso_Total=("ingreso", "sum"),
    Ingreso_Promedio=("ingreso", "mean"),
    Unidades=("unidades", "sum")
).round(0).sort_values("Ingreso_Total", ascending=False)

# Formatear numeros para la tabla
resumen["Ingreso_Total"] = resumen["Ingreso_Total"].apply(lambda x: f"${x:,.0f}")
resumen["Ingreso_Promedio"] = resumen["Ingreso_Promedio"].apply(lambda x: f"${x:,.0f}")
resumen["Unidades"] = resumen["Unidades"].astype(int)
resumen["Ventas"] = resumen["Ventas"].astype(int)

# Convertir el DataFrame a listas (que Jinja2 puede iterar)
columnas_tabla = ["Producto"] + resumen.columns.tolist()
filas_tabla = []
for producto, fila in resumen.iterrows():
    filas_tabla.append([producto] + fila.tolist())

print(f"Metricas preparadas: {len(metricas)}")
print(f"Filas de tabla: {len(filas_tabla)}")

In [ ]:
# ============================================================
# GENERAR EL GRAFICO Y GUARDARLO COMO IMAGEN
# ============================================================

# El grafico se guarda como PNG y se referencia desde el HTML
fig, ax = plt.subplots(figsize=(8, 4))

ingreso_por_producto = df_region.groupby("producto")["ingreso"].sum().sort_values()
colores = ["#93c5fd", "#60a5fa", "#3b82f6", "#2563eb"]
ingreso_por_producto.plot(kind="barh", ax=ax, color=colores)

ax.set_title(f"Ingreso por Producto — {region_reporte}", fontsize=14, fontweight="bold")
ax.set_xlabel("Ingreso Total ($)")
ax.spines[["top", "right"]].set_visible(False)  # Quitar bordes superior y derecho

# Agregar etiquetas de valor en cada barra
for i, (valor, nombre) in enumerate(zip(ingreso_por_producto, ingreso_por_producto.index)):
    ax.text(valor + 500, i, f"${valor:,.0f}", va="center", fontsize=10)

plt.tight_layout()

# Guardar en la misma carpeta que la plantilla
ruta_grafico = "grafico_reporte.png"
fig.savefig(ruta_grafico, dpi=150, bbox_inches="tight")
plt.close(fig)  # Cerrar la figura para liberar memoria

print(f"Grafico guardado: {ruta_grafico}")

In [ ]:
# ============================================================
# RENDERIZAR LA PLANTILLA CON JINJA2
# ============================================================

from jinja2 import Template

# Leer la plantilla desde disco
with open("plantilla_reporte.html", "r", encoding="utf-8") as f:
    contenido_plantilla = f.read()

# Crear el objeto Template de Jinja2
template = Template(contenido_plantilla)

# .render() reemplaza todas las variables {{ }} con los valores reales
html_final = template.render(
    titulo=f"Reporte de Ventas — {region_reporte}",
    subtitulo=f"Periodo: {periodo}",
    metricas=metricas,
    columnas_tabla=columnas_tabla,
    filas_tabla=filas_tabla,
    ruta_grafico=ruta_grafico,
    fecha_generacion=datetime.now().strftime("%d/%m/%Y %H:%M")
)

# Guardar el HTML renderizado
ruta_html = f"reporte_{region_reporte.lower()}.html"
with open(ruta_html, "w", encoding="utf-8") as f:
    f.write(html_final)

print(f"Reporte HTML generado: {ruta_html}")
print(f"Tamano: {os.path.getsize(ruta_html) / 1024:.1f} KB")

### 2.3 Usar FileSystemLoader para plantillas en disco

Cuando la plantilla hace referencia a otras plantillas (herencia, includes), es mejor usar `FileSystemLoader` en lugar de cargar el string directamente.

In [ ]:
# ============================================================
# JINJA2 CON FILESYSTEM LOADER
# ============================================================

from jinja2 import Environment, FileSystemLoader

# Environment configura donde buscar plantillas
# "." significa "carpeta actual"
env = Environment(loader=FileSystemLoader("."))

# Cargar la plantilla por nombre de archivo
template = env.get_template("plantilla_reporte.html")

# Renderizar igual que antes
html_final = template.render(
    titulo=f"Reporte de Ventas — Medellin",
    subtitulo=f"Periodo: {periodo}",
    metricas=metricas,  # Reutilizamos las metricas por simplicidad
    columnas_tabla=columnas_tabla,
    filas_tabla=filas_tabla,
    ruta_grafico=ruta_grafico,
    fecha_generacion=datetime.now().strftime("%d/%m/%Y %H:%M")
)

with open("reporte_medellin.html", "w", encoding="utf-8") as f:
    f.write(html_final)

print("Reporte generado: reporte_medellin.html")
print("Ventaja de FileSystemLoader: soporta herencia de plantillas y {% include %}")

---
## 3. Exportacion a PDF con weasyprint

weasyprint toma un HTML con CSS y lo convierte a PDF. El resultado es un documento con el mismo diseno que veriamos en el navegador, listo para imprimir o adjuntar en un correo.

In [ ]:
# ============================================================
# HTML A PDF CON WEASYPRINT
# ============================================================

from weasyprint import HTML

# Opcion A: convertir desde un archivo HTML
ruta_pdf = f"reporte_{region_reporte.lower()}.pdf"

HTML(filename=ruta_html).write_pdf(ruta_pdf)

print(f"PDF generado: {ruta_pdf}")
print(f"Tamano: {os.path.getsize(ruta_pdf) / 1024:.1f} KB")

In [ ]:
# ============================================================
# HTML A PDF DESDE STRING (sin archivo intermedio)
# ============================================================

# Si ya tenemos el HTML renderizado en memoria, podemos
# convertirlo directamente sin guardarlo primero

# base_url="." le dice a weasyprint donde buscar imagenes y CSS relativos
HTML(string=html_final, base_url=".").write_pdf("reporte_directo.pdf")

print("PDF generado desde string: reporte_directo.pdf")

In [ ]:
# ============================================================
# EMBEBER IMAGENES EN BASE64
# ============================================================

# Problema: cuando enviamos el HTML por correo o lo movemos de carpeta,
# las imagenes se pierden porque son referencias a archivos locales.
# Solucion: convertir la imagen a base64 y embeberla en el HTML.

import base64

def imagen_a_base64(ruta_imagen):
    """
    Lee una imagen y retorna un string base64 listo para usar en <img src="...">
    """
    with open(ruta_imagen, "rb") as f:
        datos = f.read()
    
    # Codificar a base64
    b64 = base64.b64encode(datos).decode("utf-8")
    
    # Detectar tipo MIME por extension
    extension = ruta_imagen.split(".")[-1].lower()
    mime = {"png": "image/png", "jpg": "image/jpeg", "jpeg": "image/jpeg"}.get(extension, "image/png")
    
    # Retornar el string completo para usar en src=""
    return f"data:{mime};base64,{b64}"

# Generar la version con imagen embebida
ruta_grafico_b64 = imagen_a_base64(ruta_grafico)

html_embebido = template.render(
    titulo=f"Reporte de Ventas — {region_reporte}",
    subtitulo=f"Periodo: {periodo}",
    metricas=metricas,
    columnas_tabla=columnas_tabla,
    filas_tabla=filas_tabla,
    ruta_grafico=ruta_grafico_b64,  # <-- Imagen en base64
    fecha_generacion=datetime.now().strftime("%d/%m/%Y %H:%M")
)

# Este HTML es autocontenido — no depende de archivos externos
HTML(string=html_embebido, base_url=".").write_pdf("reporte_autocontenido.pdf")

print("PDF autocontenido generado (imagen embebida en base64)")
print("Este archivo se puede enviar por correo sin perder el grafico.")

---
## 4. Funcion generadora de reportes

Encapsulamos todo el flujo en una sola funcion reutilizable: recibe parametros, filtra datos, genera metricas, crea el grafico, renderiza la plantilla y exporta a PDF.

In [ ]:
# ============================================================
# FUNCION COMPLETA DE GENERACION DE REPORTES
# ============================================================

def generar_reporte_pdf(
    df,
    region,
    periodo,
    plantilla_path="plantilla_reporte.html",
    carpeta_salida="reportes"
):
    """
    Genera un reporte PDF completo para una region.
    
    Parametros:
        df (DataFrame): datos de ventas
        region (str): region a reportar
        periodo (str): texto descriptivo del periodo
        plantilla_path (str): ruta a la plantilla HTML de Jinja2
        carpeta_salida (str): carpeta donde guardar los PDFs
    
    Retorna:
        str: ruta del PDF generado
    """
    # Crear carpeta de salida si no existe
    os.makedirs(carpeta_salida, exist_ok=True)
    
    # --- 1. Filtrar datos ---
    df_region = df[df["region"] == region].copy()
    
    if len(df_region) == 0:
        print(f"  Sin datos para {region}")
        return None
    
    # --- 2. Calcular metricas ---
    metricas = [
        {"valor": f"${df_region['ingreso'].sum():,.0f}", "etiqueta": "Ingreso Total"},
        {"valor": f"{len(df_region)}", "etiqueta": "Total Ventas"},
        {"valor": f"${df_region['ingreso'].mean():,.0f}", "etiqueta": "Ticket Promedio"},
        {"valor": df_region.groupby('producto')['ingreso'].sum().idxmax(), "etiqueta": "Producto Lider"},
    ]
    
    # --- 3. Tabla resumen ---
    resumen = df_region.groupby("producto").agg(
        Ventas=("ingreso", "count"),
        Ingreso_Total=("ingreso", "sum"),
        Ingreso_Promedio=("ingreso", "mean"),
        Unidades=("unidades", "sum")
    ).round(0).sort_values("Ingreso_Total", ascending=False)
    
    resumen["Ingreso_Total"] = resumen["Ingreso_Total"].apply(lambda x: f"${x:,.0f}")
    resumen["Ingreso_Promedio"] = resumen["Ingreso_Promedio"].apply(lambda x: f"${x:,.0f}")
    resumen["Unidades"] = resumen["Unidades"].astype(int)
    resumen["Ventas"] = resumen["Ventas"].astype(int)
    
    columnas = ["Producto"] + resumen.columns.tolist()
    filas = [[prod] + fila.tolist() for prod, fila in resumen.iterrows()]
    
    # --- 4. Grafico ---
    fig, ax = plt.subplots(figsize=(8, 4))
    ingreso_prod = df_region.groupby("producto")["ingreso"].sum().sort_values()
    ingreso_prod.plot(kind="barh", ax=ax, color="#2563eb")
    ax.set_title(f"Ingreso por Producto — {region}", fontsize=14, fontweight="bold")
    ax.set_xlabel("Ingreso Total ($)")
    ax.spines[["top", "right"]].set_visible(False)
    plt.tight_layout()
    
    # Guardar como temporal y convertir a base64
    ruta_tmp = os.path.join(carpeta_salida, f"tmp_grafico_{region.lower()}.png")
    fig.savefig(ruta_tmp, dpi=150, bbox_inches="tight")
    plt.close(fig)
    grafico_b64 = imagen_a_base64(ruta_tmp)
    os.remove(ruta_tmp)  # Limpiar archivo temporal
    
    # --- 5. Renderizar plantilla ---
    env = Environment(loader=FileSystemLoader("."))
    template = env.get_template(plantilla_path)
    
    html = template.render(
        titulo=f"Reporte de Ventas — {region}",
        subtitulo=f"Periodo: {periodo}",
        metricas=metricas,
        columnas_tabla=columnas,
        filas_tabla=filas,
        ruta_grafico=grafico_b64,
        fecha_generacion=datetime.now().strftime("%d/%m/%Y %H:%M")
    )
    
    # --- 6. Exportar a PDF ---
    nombre_pdf = f"reporte_{region.lower().replace(' ', '_')}.pdf"
    ruta_pdf = os.path.join(carpeta_salida, nombre_pdf)
    HTML(string=html, base_url=".").write_pdf(ruta_pdf)
    
    return ruta_pdf

In [ ]:
# ============================================================
# GENERAR REPORTES EN LOTE
# ============================================================

regiones = df["region"].unique()
archivos_generados = []

print("Generando reportes...")
print("-" * 40)

for region in regiones:
    ruta = generar_reporte_pdf(
        df=df,
        region=region,
        periodo="Enero - Junio 2024",
        carpeta_salida="reportes"
    )
    
    if ruta:
        tamano = os.path.getsize(ruta) / 1024
        print(f"  {ruta} ({tamano:.1f} KB)")
        archivos_generados.append(ruta)

print("-" * 40)
print(f"Total: {len(archivos_generados)} reportes generados en /reportes")

---
## 5. Automatizacion y distribucion

Con la funcion lista, el ultimo paso es hacer que se ejecute sola — sin que alguien abra Jupyter y presione "Run All".

### 5.1 Crear el script ejecutable

In [ ]:
# ============================================================
# CREAR SCRIPT .py INDEPENDIENTE
# ============================================================

# Este script se puede ejecutar desde la terminal con:
# python generar_reportes.py

script = '''
#!/usr/bin/env python3
"""
Script de generacion automatica de reportes.
Uso: python generar_reportes.py
"""

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import os
import base64
from datetime import datetime
from jinja2 import Environment, FileSystemLoader
from weasyprint import HTML


# --- Configuracion ---
ARCHIVO_DATOS = "ventas_completas.csv"
PLANTILLA = "plantilla_reporte.html"
CARPETA_SALIDA = "reportes"
PERIODO = "Enero - Junio 2024"


def imagen_a_base64(ruta):
    with open(ruta, "rb") as f:
        datos = f.read()
    b64 = base64.b64encode(datos).decode("utf-8")
    return f"data:image/png;base64,{b64}"


def generar_reporte(df, region, periodo, plantilla_path, carpeta_salida):
    os.makedirs(carpeta_salida, exist_ok=True)
    df_r = df[df["region"] == region].copy()
    if len(df_r) == 0:
        return None

    metricas = [
        {"valor": f"${df_r['ingreso'].sum():,.0f}", "etiqueta": "Ingreso Total"},
        {"valor": f"{len(df_r)}", "etiqueta": "Total Ventas"},
        {"valor": f"${df_r['ingreso'].mean():,.0f}", "etiqueta": "Ticket Promedio"},
        {"valor": df_r.groupby("producto")["ingreso"].sum().idxmax(), "etiqueta": "Producto Lider"},
    ]

    resumen = df_r.groupby("producto").agg(
        Ventas=("ingreso", "count"),
        Ingreso_Total=("ingreso", "sum"),
        Ingreso_Promedio=("ingreso", "mean"),
        Unidades=("unidades", "sum")
    ).round(0).sort_values("Ingreso_Total", ascending=False)

    resumen["Ingreso_Total"] = resumen["Ingreso_Total"].apply(lambda x: f"${x:,.0f}")
    resumen["Ingreso_Promedio"] = resumen["Ingreso_Promedio"].apply(lambda x: f"${x:,.0f}")
    resumen["Unidades"] = resumen["Unidades"].astype(int)
    resumen["Ventas"] = resumen["Ventas"].astype(int)

    columnas = ["Producto"] + resumen.columns.tolist()
    filas = [[p] + f.tolist() for p, f in resumen.iterrows()]

    fig, ax = plt.subplots(figsize=(8, 4))
    df_r.groupby("producto")["ingreso"].sum().sort_values().plot(
        kind="barh", ax=ax, color="#2563eb"
    )
    ax.set_title(f"Ingreso por Producto — {region}")
    ax.set_xlabel("Ingreso ($)")
    ax.spines[["top", "right"]].set_visible(False)
    plt.tight_layout()
    tmp = os.path.join(carpeta_salida, "tmp.png")
    fig.savefig(tmp, dpi=150, bbox_inches="tight")
    plt.close(fig)
    grafico_b64 = imagen_a_base64(tmp)
    os.remove(tmp)

    env = Environment(loader=FileSystemLoader("."))
    html = env.get_template(plantilla_path).render(
        titulo=f"Reporte de Ventas — {region}",
        subtitulo=f"Periodo: {periodo}",
        metricas=metricas,
        columnas_tabla=columnas,
        filas_tabla=filas,
        ruta_grafico=grafico_b64,
        fecha_generacion=datetime.now().strftime("%d/%m/%Y %H:%M")
    )

    ruta_pdf = os.path.join(carpeta_salida, f"reporte_{region.lower().replace(' ', '_')}.pdf")
    HTML(string=html, base_url=".").write_pdf(ruta_pdf)
    return ruta_pdf


if __name__ == "__main__":
    print(f"[{datetime.now():%Y-%m-%d %H:%M}] Iniciando generacion de reportes")
    
    df = pd.read_csv(ARCHIVO_DATOS, parse_dates=["fecha"])
    df["ingreso"] = df["unidades"] * df["precio_unitario"]
    
    regiones = df["region"].unique()
    generados = 0
    
    for region in regiones:
        ruta = generar_reporte(df, region, PERIODO, PLANTILLA, CARPETA_SALIDA)
        if ruta:
            print(f"  {ruta}")
            generados += 1
    
    print(f"[{datetime.now():%Y-%m-%d %H:%M}] Listo. {generados} reportes en /{CARPETA_SALIDA}")
'''

with open("generar_reportes.py", "w", encoding="utf-8") as f:
    f.write(script)

print("Script creado: generar_reportes.py")
print("Ejecutar con: python generar_reportes.py")

### 5.2 Programar ejecucion con cron (Linux/Mac) o Task Scheduler (Windows)

In [ ]:
# ============================================================
# CRON — ejecutar el script automaticamente
# ============================================================

# cron es el programador de tareas de Linux/Mac
# Se configura editando el archivo crontab

# Para abrir el editor de crontab desde la terminal:
# crontab -e

# Formato de una linea cron:
# minuto hora dia_mes mes dia_semana comando
#   0-59  0-23  1-31  1-12  0-7(0=dom)

# Ejemplos:

# Ejecutar todos los lunes a las 7:00 AM
cron_lunes = "0 7 * * 1 cd /ruta/al/proyecto && python generar_reportes.py >> /var/log/reportes.log 2>&1"

# Ejecutar el primer dia de cada mes a las 6:00 AM
cron_mensual = "0 6 1 * * cd /ruta/al/proyecto && python generar_reportes.py >> /var/log/reportes.log 2>&1"

# Ejecutar cada 6 horas
cron_6h = "0 */6 * * * cd /ruta/al/proyecto && python generar_reportes.py"

print("Ejemplos de configuracion cron:")
print()
print("Lunes 7:00 AM:")
print(f"  {cron_lunes}")
print()
print("Primer dia del mes 6:00 AM:")
print(f"  {cron_mensual}")
print()
print("Cada 6 horas:")
print(f"  {cron_6h}")
print()
print("NOTA: >> redirige la salida a un archivo de log")
print("      2>&1 redirige errores al mismo log")
print("      cd ... && asegura que el script corre en la carpeta correcta")

### 5.3 Envio por correo

In [ ]:
# ============================================================
# ENVIAR REPORTE POR CORREO — smtplib
# ============================================================

# smtplib viene con Python — no requiere instalacion
# Este ejemplo muestra la estructura completa
# Para que funcione, se necesitan credenciales SMTP reales

import smtplib
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
from email.mime.application import MIMEApplication

def enviar_reporte_por_correo(
    destinatario,
    asunto,
    cuerpo_html,
    ruta_pdf,
    remitente,
    password,
    servidor_smtp="smtp.gmail.com",
    puerto=587
):
    """
    Envia un correo con el reporte PDF adjunto.
    
    Parametros:
        destinatario (str): correo del destinatario
        asunto (str): asunto del correo
        cuerpo_html (str): cuerpo del correo en HTML
        ruta_pdf (str): ruta al PDF a adjuntar
        remitente (str): correo del remitente
        password (str): contrasena o app password
        servidor_smtp (str): servidor SMTP
        puerto (int): puerto SMTP
    """
    # Crear el mensaje
    msg = MIMEMultipart()
    msg["From"] = remitente
    msg["To"] = destinatario
    msg["Subject"] = asunto
    
    # Agregar cuerpo HTML
    msg.attach(MIMEText(cuerpo_html, "html"))
    
    # Adjuntar el PDF
    with open(ruta_pdf, "rb") as f:
        adjunto = MIMEApplication(f.read(), _subtype="pdf")
        nombre_archivo = os.path.basename(ruta_pdf)
        adjunto.add_header("Content-Disposition", "attachment", filename=nombre_archivo)
        msg.attach(adjunto)
    
    # Enviar
    with smtplib.SMTP(servidor_smtp, puerto) as server:
        server.starttls()               # Iniciar conexion segura
        server.login(remitente, password)  # Autenticar
        server.send_message(msg)         # Enviar
    
    print(f"  Correo enviado a {destinatario}")


# --- Ejemplo de uso (NO ejecutar sin credenciales reales) ---

# Para Gmail, se necesita un "App Password":
# 1. Activar verificacion en 2 pasos en la cuenta de Google
# 2. Ir a myaccount.google.com > Seguridad > Contraseñas de aplicacion
# 3. Generar una contrasena para "Correo" en "Otro dispositivo"

print("Funcion enviar_reporte_por_correo() lista.")
print("Requiere credenciales SMTP para ejecutar.")
print()
print("Ejemplo de uso:")
print('''  enviar_reporte_por_correo(''')
print('''      destinatario="gerente@empresa.com",''')
print('''      asunto="Reporte Semanal de Ventas - Bogota",''')
print('''      cuerpo_html="<h3>Adjunto el reporte de la semana.</h3>",''')
print('''      ruta_pdf="reportes/reporte_bogota.pdf",''')
print('''      remitente="reportes@empresa.com",''')
print('''      password="xxxx xxxx xxxx xxxx"''')
print('''  )''')

---
## Resumen de la subseccion

| Herramienta | Que hace | Cuando usarla |
|---|---|---|
| **Papermill** | Ejecuta notebooks con parametros distintos | Reportes tecnicos, analisis parametrizados, ejecucion en lote |
| **Jinja2** | Renderiza plantillas HTML con datos dinamicos | Reportes profesionales con diseno personalizado |
| **weasyprint** | Convierte HTML+CSS a PDF | Cuando el destinatario necesita un PDF |
| **base64** | Embebe imagenes en el HTML | Para que el reporte sea autocontenido |
| **cron** | Programa ejecucion automatica | Reportes recurrentes (semanal, mensual) |
| **smtplib** | Envia correos con adjuntos | Distribucion automatica por correo |

### Flujo completo

1. Datos llegan (CSV, BD, API)
2. Script Python filtra, agrega y genera graficos
3. Jinja2 inyecta los datos en una plantilla HTML
4. weasyprint convierte el HTML a PDF
5. smtplib envia el PDF por correo
6. cron ejecuta todo de forma automatica

### Siguiente paso
En la **Unidad 2** analizaremos las caracteristicas que hacen que un producto de datos sea confiable: calidad, exploracion visual con Orange, KPIs y storytelling.